[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Radar_Signal_Processing.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Radar Signal Processing

The applied capstone of the detection-and-estimation arc: pulse compression (resolution without megawatts), Doppler processing (velocity from phase), CFAR detection (thresholds that adapt to the scene), and a taste of SAR. We build a complete pulse-Doppler radar in NumPy and verify every extracted target parameter against the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb) S4 (matched filters/ROC), [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (chirps, FFT), [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

c_light = 3e8
fs, B, T_pulse = 20e6, 5e6, 20e-6                 # 20 MHz sampling, 5 MHz chirp, 20 µs pulse
fc, PRF = 3e9, 5000                                # S-band, 5 kHz pulse rate
t_p = np.arange(0, T_pulse, 1/fs)
chirp_tx = np.exp(1j*np.pi*(B/T_pulse)*(t_p - T_pulse/2)**2)   # LFM pulse

---
### 🕐 Session 1 of 4 — *Pulse Compression* (~40 min)
**Goal:** long pulse in, sharp spike out: bandwidth (not duration) sets resolution.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 2 (Doppler).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Pulse Compression</b></summary>

**Timing (~40 min).** 10 min the dilemma · 10 min the chirp's resolution · 12 min the demo · 8 min the time–bandwidth product.

**Board first — state the dilemma as genuinely impossible, then resolve it.** Range resolution wants a *short* pulse, because two echoes must not overlap. Detection range wants *energy*, and energy is power × duration, so it wants a *long* pulse. Peak power is capped by the transmitter hardware. Ask the room how to escape. Most propose more power; the answer is that you do not have to choose, because resolution does not actually depend on duration.

**The reframe that carries the session.** $\Delta R = c/2B$ — resolution is set by **bandwidth**, not duration. An unmodulated pulse has $B \approx 1/T$, which is why short-pulse radar exists at all, but a *chirp* decouples the two: sweep the frequency across the pulse and you get 5 MHz of bandwidth from a 20 µs pulse. Match-filter on receive and it collapses to a spike of width $1/B$. You transmitted 20 µs of energy and received the resolution of a 0.2 µs pulse.

**Put the numbers on the board — they are startling.** The raw pulse spans $cT/2 = 3000$ m of range. After compression the resolution is $c/2B = 30$ m. That is a factor of 100, which is exactly the time–bandwidth product $BT = 5\times10^6 \times 20\times10^{-6} = 100$. The compression gain is not a tuning parameter; it *is* $BT$, and students should be able to state that from the two numbers.

**Ask the room.** "Why is the matched filter the right receive processing?" Because it maximises output SNR in white noise — the theorem from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. Radar is the cleanest possible application of that result: you know exactly what you transmitted, so you know exactly what to correlate against. `np.correlate(rx, chirp_tx, "valid")` is that theorem, one line.

**Set the demo up as a test, not a display.** Two targets 75 m apart, resolution 30 m — so they *should* separate, and the assert requires the estimates to land within one resolution cell. Ask what would happen at 20 m separation before running it: they would merge into one peak, and no amount of processing would recover them, because the bandwidth is simply insufficient. Being able to predict success or failure from $c/2B$ is the takeaway.

**Mention sidelobes if you have time.** The compressed pulse has range sidelobes at roughly −13 dB for an unwindowed LFM, visible on the plot below the peaks. A strong target's sidelobes can mask a weak one nearby, which is why real radars window the reference chirp — trading main-lobe width for sidelobe suppression, exactly the trade from [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb)'s window discussion.
</details>

## 2. The Chirp's Bargain

💡 **Intuition.** Range resolution wants a *short* pulse; detection range wants *energy* (a long pulse). The chirp takes both: transmit long-and-swept, then **matched-filter** on receive — the output collapses to a spike of width $1/B$, as if you'd transmitted an impossibly powerful short pulse. Resolution comes from **bandwidth**, not duration: $\Delta R = c/2B$. The compression gain is the time–bandwidth product $BT$ — here ×100.

In [2]:
# two targets 75 m apart — the RAW 20 µs pulse spans 3 km of range; compression resolves them
R1, R2 = 3000.0, 3075.0
delay = lambda R: int(round(2*R/c_light * fs))
n_rx = 6000
rx = np.zeros(n_rx, complex)
for R, amp in [(R1, 1.0), (R2, 0.7)]:
    d = delay(R)
    rx[d:d+len(chirp_tx)] += amp * chirp_tx
rx += 0.1*(rng.standard_normal(n_rx) + 1j*rng.standard_normal(n_rx))

compressed = np.abs(np.correlate(rx, chirp_tx, "valid"))
rng_axis = np.arange(len(compressed)) * c_light/(2*fs)

plt.figure(figsize=(9, 2.6))
plt.plot(rng_axis, 20*np.log10(compressed/compressed.max() + 1e-6))
for R in (R1, R2): plt.axvline(R, color="r", linestyle=":", linewidth=0.8)
plt.xlim(2900, 3200); plt.ylim(-40, 2)
plt.xlabel("range [m]"); plt.ylabel("dB")
plt.title(f"pulse compression: {c_light/2/B:.0f} m resolution from a pulse that spans {c_light*T_pulse/2:.0f} m raw")
plt.tight_layout(); plt.show()

peaks = sig.find_peaks(compressed, height=compressed.max()*0.3, distance=8)[0]
est = rng_axis[peaks]
print(f"planted ranges: {R1:.0f}, {R2:.0f} m   estimated: {est.round(0)}")
assert np.abs(np.sort(est)[:2] - [R1, R2]).max() < c_light/(2*B)

planted ranges: 3000, 3075 m   estimated: [3000. 3075.]


/tmp/ipykernel_2990837/1939572244.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Both targets recovered at **3000 and 3075 m**, matching the planted truth exactly to the printed precision — from a pulse whose raw extent covers 3000 m of range.

That last point is the one worth dwelling on. The transmitted pulse is 20 µs long, which occupies $cT/2 = 3000$ m of range. Two echoes 75 m apart overlap almost completely on receive; nothing in the raw data looks like two targets. Matched filtering collapses each echo to a spike of width $c/2B = 30$ m, and 75 m separation then resolves comfortably.

**Resolution comes from bandwidth, not duration.** This is the central fact of the session, and the two numbers make it concrete: raw extent 3000 m, compressed resolution 30 m, a factor of **100**. That factor is exactly the time–bandwidth product $BT = 5\,\text{MHz} \times 20\,\mu\text{s} = 100$. The compression gain is not tunable — it is $BT$, and knowing that lets you design a pulse from requirements rather than by trial.

The bargain is worth stating plainly. Detection range needs *energy*, and energy is power × duration, so it wants a long pulse. Range resolution needs echoes not to overlap, so it wants a short one. Peak transmitter power is fixed by hardware. The chirp escapes the trade by sweeping frequency across a long pulse: you transmit 20 µs of energy and receive the resolution of a 0.2 µs pulse. That is why every modern radar transmits chirps.

**And the processing is a theorem you already proved.** `np.correlate(rx, chirp_tx, "valid")` is the matched filter, which maximises output SNR in white noise — the result from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. Radar is its cleanest application: you know precisely what you transmitted, so you know precisely what to correlate against.

**Two things to notice on the plot.** The dashed truth markers sit on the peaks, and the `assert` enforces agreement to better than one resolution cell — so this is a checked claim, not a visual impression. And look below the peaks: the compressed pulse has **range sidelobes**, around −13 dB for an unwindowed chirp. A strong target's sidelobes can bury a weak target nearby, which is why production radars window the reference chirp, accepting a wider main lobe to push the sidelobes down. Same trade as windowing anywhere else in [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb).

Try moving the targets to 20 m apart and re-running: they merge into a single peak, and no processing recovers them. Below $c/2B$, the information is not there.

---
### 🕐 Session 2 of 4 — *Doppler Processing* (~40 min)
**Goal:** velocity from pulse-to-pulse phase: the range-Doppler map, verified against planted targets.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (CFAR).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Doppler Processing</b></summary>

**Timing (~40 min).** 10 min why one pulse cannot measure velocity · 12 min slow time and the pulse-to-pulse phase · 10 min the range–Doppler map · 8 min resolution and ambiguity.

**Board first — the two time axes.** Draw the data matrix: rows are pulses, columns are range samples. **Fast time** (along a row) is range; **slow time** (down a column) is pulse index. That picture is the session — everything else is "FFT down the columns."

**Make the phase argument concrete with numbers.** A target at 30 m/s moves 6 mm between pulses at 5 kHz PRF. Ask whether that is measurable in *range*: no — the range bin is 7.5 m, so the target does not move even a thousandth of a cell. Then ask about *phase*: the wavelength is 10 cm, so 6 mm of range change is 12 mm of two-way path, which is over 40° of phase. **Range is blind to it, phase is loud.** That contrast is why coherent radar exists, and it is the moment the session turns.

**Then the reframe.** Each range cell, read down the pulses, contains a slow-time sinusoid whose frequency *is* the Doppler shift. So velocity estimation is spectral estimation, and an FFT down each column produces the range–Doppler map. Nothing new is needed — the room has been doing FFTs for the whole curriculum.

**Point at the clutter ridge and say why it matters.** The strong stationary return at 4000 m is 6× the amplitude of the targets — in real scenes, ground clutter can be 60 dB above the aircraft you want. In *range* it is inseparable from a target at the same range. In Doppler it sits at exactly 0 m/s while movers sit elsewhere, so a simple notch removes it. This is the entire reason pulse-Doppler radar can see a car moving in front of a mountain, and it is worth stating as the practical payoff of the session.

**Do the resolution arithmetic before reading the estimates.** Velocity bin width is $\lambda/(2 N T_{PRI}) = 0.1/(2 \cdot 64 \cdot 200\,\mu s) = 3.9$ m/s. So the estimates are quantised to a 3.9 m/s grid, and 30 m/s must land on the bin at 31.25. Have the room predict the printed values *before* running — getting +31.2 and −15.6 from theory rather than from the output is the most satisfying moment available here.

**Then the follow-up that generalises.** More pulses give finer Doppler resolution ($N$ in the denominator), but $N T_{PRI}$ is the coherent processing interval, during which the target must not change range cell or accelerate much. Resolution costs dwell time — the same time–frequency trade as every spectral estimate in the curriculum, with a physical constraint attached.

**Mention ambiguity if time allows.** PRF sets both the unambiguous range ($c/2\text{PRF} = 30$ km) and the unambiguous velocity ($\pm\lambda \text{PRF}/4 = \pm125$ m/s). Raising PRF helps one and hurts the other, which is why real radars stagger PRFs and resolve ambiguities across them.
</details>

## 3. Velocity Is a Phase Story

💡 **Intuition.** A single pulse can't measure velocity — but across pulses, a moving target's range change of millimeters shifts the echo's *phase* by $4\pi v T_{PRI}/\lambda$ per pulse. Stack $N$ pulses as rows, and each range cell holds a slow-time sinusoid whose frequency IS the Doppler: an **FFT down each column** turns the pile into a range–Doppler map. Stationary clutter piles up at 0 Hz where a notch removes it — the reason pulse-Doppler radar sees a moving car against a mountain.

In [3]:
# 64-pulse coherent interval; targets: (3000 m, +30 m/s), (5000 m, −15 m/s), clutter at 4000 m
n_pulses, lam = 64, c_light/fc
PRI = 1/PRF
targets = [(3000, 30.0, 1.0), (5000, -15.0, 0.8)]
clutter = (4000, 0.0, 6.0)                              # strong stationary return

data = np.zeros((n_pulses, 5000), complex)
for p in range(n_pulses):
    for R0, v, amp in targets + [clutter]:
        R = R0 + v * p * PRI
        d = int(round(2*R/c_light * fs))
        phase = np.exp(-1j*4*np.pi*R/lam)
        data[p, d:d+len(chirp_tx)] += amp * phase * chirp_tx
    data[p] += 0.15*(rng.standard_normal(5000) + 1j*rng.standard_normal(5000))

# compress each pulse, then FFT across pulses
comp = np.stack([np.correlate(row, chirp_tx, "valid") for row in data])
rd_map = np.fft.fftshift(np.fft.fft(comp, axis=0), axes=0)
vel_axis = -np.fft.fftshift(np.fft.fftfreq(n_pulses, PRI)) * lam/2   # e^{-j4πR/λ}: +v ⇒ negative FFT bin
rng_axis2 = np.arange(comp.shape[1]) * c_light/(2*fs)

plt.figure(figsize=(8.5, 3.4))
plt.pcolormesh(rng_axis2/1000, vel_axis, 20*np.log10(np.abs(rd_map)+1e-3), shading="auto", vmin=0, vmax=60)
plt.colorbar(label="dB"); plt.xlabel("range [km]"); plt.ylabel("velocity [m/s]")
plt.title("range–Doppler map: movers separate from the clutter ridge at v=0")
plt.tight_layout(); plt.show()

# ORACLE: extract peaks (excluding the v≈0 clutter line) and compare to planted truth
mask = np.abs(vel_axis)[:, None] > 3
mag = np.abs(rd_map) * mask
for R_t, v_t, _ in targets:
    i, j = np.unravel_index(np.argmax(mag), mag.shape)
    print(f"detected: R = {rng_axis2[j]:.0f} m, v = {vel_axis[i]:+.1f} m/s   (planted {R_t} m, {v_t:+.1f} m/s)")
    mag[max(0,i-3):i+4, max(0,j-8):j+8] = 0

detected: R = 3000 m, v = +31.2 m/s   (planted 3000 m, +30.0 m/s)
detected: R = 5002 m, v = -15.6 m/s   (planted 5000 m, -15.0 m/s)


/tmp/ipykernel_2990837/579600621.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Both movers extracted: **3000 m at +31.2 m/s** and **5002 m at −15.6 m/s**, against planted values of 3000 m / +30 m/s and 5000 m / −15 m/s. And the apparent errors are not estimation error at all — they are the grid.

**Work the resolution out and the "errors" disappear.** The velocity bin width is
$$\Delta v = \frac{\lambda}{2 N T_{PRI}} = \frac{0.1}{2 \cdot 64 \cdot 200\,\mu\text{s}} = 3.906 \text{ m/s}.$$
The planted +30 m/s falls at bin index $30/3.906 = 7.68$, so the nearest bin is 8, at **+31.25 m/s**. The planted −15 sits at index −3.84, nearest bin −4, at **−15.625 m/s**. Those are exactly the printed values. The estimator did not approximate anything — it returned the correct bin, and the offset is quantisation of a continuous velocity onto a discrete FFT grid.

The same holds in range: samples are spaced $c/2f_s = 7.5$ m apart, so 5002 and 5000 are the same range cell. **Both estimates are exact to the resolution of the measurement**, which is a much stronger statement than "close to the truth."

**Why velocity is a phase story.** At 30 m/s and a 5 kHz PRF, the target moves 6 mm between pulses. The range bin is 7.5 m, so in *range* that motion is undetectable — not merely small, but a thousandth of a cell. But the wavelength is 10 cm, and 6 mm of range change is 12 mm of two-way path, which is over 40° of phase. Range is blind to the motion; phase is shouting about it. Coherent radar exists to exploit exactly that asymmetry.

Stack the pulses and each range cell, read down the column, holds a slow-time sinusoid whose frequency *is* the Doppler shift. So velocity estimation is spectral estimation, and `np.fft.fft(comp, axis=0)` is the whole of it. Two time axes — fast time along a row for range, slow time down a column for velocity — and one familiar transform.

**The clutter ridge is the practical payoff.** The stationary return at 4000 m is 6× the amplitude of either target, and in *range alone* it is inseparable from anything at the same distance. In the Doppler dimension it collapses onto the $v = 0$ line while the movers sit at ±4 bins away, so a notch at zero velocity removes it entirely. That is why a pulse-Doppler radar can track a car driving in front of a mountain, and why real ground clutter — often 60 dB above the target — is a solvable problem rather than a fatal one. Note the extraction code does exactly this: `mask = np.abs(vel_axis)[:, None] > 3` excludes the zero-Doppler line before searching for peaks.

**The cost of resolution.** Finer velocity resolution means larger $N$, and $N T_{PRI}$ is the coherent processing interval — 12.8 ms here. Throughout that dwell the target must stay in its range cell and not accelerate appreciably, or the slow-time sinusoid smears. Resolution is bought with dwell time, the same time–frequency trade as every spectral estimate in this curriculum, now with a physical constraint attached to it.

---
### 🕐 Session 3 of 4 — *CFAR Detection* (~35 min)
**Goal:** thresholds that ride the local noise: constant false-alarm rate in inhomogeneous scenes.
**Builds on:** Session 2; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 4 (SAR at a glance).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: CFAR Detection</b></summary>

**Timing (~35 min).** 8 min why a fixed threshold cannot work · 10 min the CA-CFAR window · 10 min the demo · 7 min the failure modes.

**Board first — kill the fixed threshold with a dilemma.** Draw a range profile with a quiet region and a cluttered region. Set the threshold for the quiet zone and the clutter zone floods with false alarms; set it for the clutter and every quiet-zone target is missed. There is no single number that works, and the room should see that before CFAR is introduced. The scene is *inhomogeneous*, so the threshold must be too.

**Then the name, unpacked.** Constant False Alarm Rate: the goal is not a constant *threshold* but a constant false-alarm *rate*, achieved by letting the threshold ride the local noise. Students often read CFAR as a thresholding trick; it is better understood as normalising by a local noise estimate so the detection statistic has the same distribution everywhere.

**Draw the window, and make the guard cells earn their name.** Cell under test in the middle, guard cells immediately around it, reference cells beyond. Ask why the guard cells exist. Because a target spans more than one cell — after pulse compression it occupies a few — so without a gap, the target's own energy enters the noise estimate, raises the threshold, and helps hide itself. That self-masking is the failure the guard band prevents, and it is worth making someone say it aloud.

**Frame the result as a rate, since that is what is claimed.** 1 false alarm from CFAR against 109 from the fixed threshold, across a 9× noise step. Point out that the *fixed* threshold's failure is entirely in the clutter zone — it was set correctly for the quiet region and is simply wrong everywhere else. And note honestly that CFAR's single false alarm at cell 60 is not a defect: with `scale=9.0` on exponential noise a small false-alarm probability per cell times 800 cells gives you order-1 false alarms. The name promises a constant rate, not zero.

**Ask the room.** "What sets `scale`?" It is chosen for a target false-alarm probability given the noise statistics — for exponential (square-law detected) noise there is a closed-form relation between the multiplier, the number of reference cells, and $P_{FA}$. Raising it trades detections for fewer false alarms. This is the ROC curve from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 appearing as a single tunable constant, and it is worth naming that connection.

**Give the failure modes airtime — this is where practitioners live.** CA-CFAR averages its reference cells, so it breaks in two well-known ways. *Multiple targets*: a second target inside the reference window inflates the noise estimate and masks the first — the fix is GO/SO-CFAR or ordered-statistic CFAR, which use max/min or a percentile rather than a mean. *Clutter edges*: at the step in this very demo, a window straddling the boundary averages two different noise levels and behaves badly on both sides. Have the room look at the threshold trace right around cell 400; the transition is visible.

**Close on the framing.** Detection is not a fixed comparison but an estimate-then-decide procedure, and half the engineering is estimating what "normal" looks like locally. That idea generalises well beyond radar — it is the same logic as adaptive thresholding in image processing and local baselining in anomaly detection.
</details>

## 4. The Adaptive Threshold

💡 **Intuition.** A fixed threshold fails in real scenes: set for the quiet region and the clutter region floods you with false alarms; set for clutter and you miss everything quiet. **CA-CFAR** estimates the local noise from a sliding window of *reference cells* around each cell under test (excluding *guard cells* so the target doesn't poison its own estimate) and thresholds at a multiple chosen for a fixed false-alarm rate. The threshold *rides the terrain*.

In [4]:
# 1-D range profile with a noise step (quiet region | clutter region) and 3 targets
n_cells = 800
noise_level = np.where(np.arange(n_cells) < 400, 1.0, 8.0)
profile = noise_level * rng.exponential(1.0, n_cells)         # square-law detected noise
target_cells = [120, 300, 610]
for tc, amp in zip(target_cells, [18, 14, 90]):
    profile[tc] += amp

def ca_cfar(x, n_ref=16, n_guard=2, scale=9.0):
    th = np.full_like(x, np.inf)
    for i in range(n_ref+n_guard, len(x)-n_ref-n_guard):
        ref = np.r_[x[i-n_ref-n_guard:i-n_guard], x[i+n_guard+1:i+n_guard+1+n_ref]]
        th[i] = scale * ref.mean()
    return th

th = ca_cfar(profile)
fixed_th = 9.0 * profile[:400].mean()                        # fixed threshold set in the quiet zone

det_cfar = np.where(profile > th)[0]
det_fixed = np.where(profile > fixed_th)[0]
plt.figure(figsize=(9.5, 2.8))
plt.semilogy(profile, linewidth=0.6, label="range profile")
plt.semilogy(th, "r", linewidth=1, label="CFAR threshold (rides the step)")
plt.axhline(fixed_th, color="gray", linestyle="--", linewidth=1, label="fixed threshold")
for tc in target_cells: plt.axvline(tc, color="g", linestyle=":", linewidth=0.8)
plt.legend(fontsize=7); plt.title("noise step at cell 400: fixed threshold drowns, CFAR adapts")
plt.tight_layout(); plt.show()
print(f"targets at {target_cells}")
print(f"CFAR detections:  {[int(i) for i in det_cfar]}  (false alarms: {len(set(det_cfar)-set(target_cells))})")
print(f"fixed-threshold false alarms in the clutter zone: {int((det_fixed >= 400).sum() - 1)}")

targets at [120, 300, 610]
CFAR detections:  [60, 120, 300, 610]  (false alarms: 1)
fixed-threshold false alarms in the clutter zone: 109


/tmp/ipykernel_2990837/842502740.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** All three targets found, with **1** false alarm from CFAR against **109** from the fixed threshold — across a scene whose noise level steps by 9× at cell 400.

**Read the fixed threshold's failure precisely.** It was not set badly; it was set *correctly for the quiet region*, using `profile[:400].mean()`. It then produced 109 false alarms in the clutter region, because a threshold calibrated for one noise level is meaningless at another. And the dilemma has no fixed-number solution: raise it to survive the clutter and the quiet-zone targets at cells 120 and 300 disappear. **No single threshold works on an inhomogeneous scene**, which is the entire argument for the session.

CFAR escapes by estimating the noise *locally* — a sliding window of reference cells around each cell under test — and thresholding at a multiple of that estimate. The red trace visibly steps up at cell 400, tracking the terrain. The name is worth taking literally: the goal is not a constant threshold but a constant false-alarm *rate*, achieved by normalising away the local noise level so the detection statistic has the same distribution everywhere.

**The guard cells are the subtle part.** After pulse compression a target occupies several adjacent cells. Without a gap between the cell under test and the reference window, the target's own energy would enter its noise estimate, raise its own threshold, and help conceal it — self-masking. `n_guard=2` is what prevents that, and it is a genuine design parameter rather than defensive padding.

**And CFAR's own false alarm is not a defect.** One detection at cell 60 with no target there is exactly what "constant false alarm *rate*" promises: with `scale=9.0` against exponential noise, the per-cell false-alarm probability is small but nonzero, and across ~770 tested cells you should expect of order one. Reporting zero would actually be suspicious — it would mean the threshold was set far too high and detections were being lost. The multiplier is the knob that trades detections against false alarms, which is the ROC curve from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 compressed into a single constant.

**Two failure modes worth knowing, both live in this demo.** CA-CFAR *averages* its reference cells, so a second target inside the reference window inflates the noise estimate and can mask the first — the reason GO/SO-CFAR and ordered-statistic CFAR exist, using max, min, or a percentile instead of a mean. And at a **clutter edge**, a window straddling the boundary averages two different noise levels and misbehaves on both sides; look closely at the threshold trace either side of cell 400 and the transition region is visible.

The generalisable idea: detection is not a comparison against a fixed number, it is *estimate the local normal, then decide*. That structure recurs well outside radar — adaptive thresholding in image processing, local baselining in anomaly detection — wherever "normal" is not constant across the data.

---
### 🕐 Session 4 of 4 — *Synthetic Aperture at a Glance* (~30 min)
**Goal:** how a small antenna on a moving platform becomes a huge one: SAR in one simulation.
**Builds on:** Sessions 1–3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Synthetic Aperture at a Glance</b></summary>

**Timing (~30 min).** 8 min the aperture argument · 10 min why the along-track signal is a chirp · 8 min the demo · 4 min the punchline about resolution.

**Board first — start from a limit the room already accepts.** [Array Processing](./Array_Processing.ipynb) established that angular resolution scales with aperture size. A satellite antenna is a few metres across, which at these wavelengths gives hopeless azimuth resolution from orbit — kilometres. Ask how to get a kilometre-wide antenna into space. You do not: you **fly** a small one and record coherently along the path. The aperture is synthesised from positions rather than built from metal.

**The key structural insight, and it should feel familiar.** As the platform flies past a scatterer, the range follows $R(u) = \sqrt{R_0^2 + (u-x_0)^2}$, which near closest approach is approximately quadratic in $u$. Phase is $-4\pi R/\lambda$, so phase is quadratic in position — and a signal with quadratic phase is a **chirp**. Azimuth compression is therefore Session 1's matched filter, applied along the track instead of along fast time. Say this explicitly: *the same operation, rotated 90°*. Students who see SAR as a new subject find it hard; students who see it as pulse compression on a second axis find it straightforward.

**Then the punchline, which is genuinely counterintuitive — set it up carefully.** Ask what happens to SAR azimuth resolution as the platform flies *further away*. The instinct is that resolution gets worse. It does not: at greater range the platform stays within the antenna beam for longer, so the synthetic aperture is correspondingly longer, and the two effects cancel almost exactly. Strip-map SAR azimuth resolution is approximately $D/2$ — **half the physical antenna length** — independent of range. A *smaller* antenna gives *better* azimuth resolution, because it has a wider beam and therefore a longer synthetic aperture. This is the most surprising fact in the session and worth the time to land properly.

**Point at the honest simplifications.** The demo compresses in azimuth only, at a single range, with a matched filter built for a scatterer at $x = 0$ — so range–azimuth coupling, range cell migration correction, and platform motion errors are all absent. Real SAR processors (range–Doppler, chirp scaling, back-projection) exist mostly to handle exactly those. Say so; the cell demonstrates the principle, and the engineering around it is a field of its own.

**Ask the room.** "Why does this require *coherent* recording?" Because the entire method rests on phase across positions. Lose phase coherence — through platform position error, oscillator drift, or atmospheric variation — and the synthetic aperture collapses to the physical one. This is why SAR platforms carry precise navigation and why autofocus algorithms exist. It is also the same coherence requirement as Session 2's Doppler processing, on a spatial axis.

**Close the workshop.** Bandwidth buys range resolution, pulse-to-pulse phase buys velocity, local estimation buys detection, and motion buys aperture. Four sessions, four resources, one matched filter appearing in three of them.
</details>

## 5. SAR: The Aperture You Fly

💡 **Intuition.** [Array resolution](./Array_Processing.ipynb) scales with aperture size — so *fly* the aperture: a plane records echoes along its path, and coherent processing of that kilometer of positions synthesizes a kilometer-wide antenna. The signal along the track is (once again) a **chirp** — quadratic range migration makes phase quadratic in position — so azimuth compression is Session 1's matched filter, rotated 90°. Range chirp + azimuth chirp = imagery from orbit.

In [5]:
# strip-map SAR toy: 3 point scatterers, platform flying past — azimuth compression
R0 = 5000.0                                              # closest range
n_pos, du = 512, 0.4                                     # 0.4 m between pulses → 205 m aperture
u = (np.arange(n_pos) - n_pos/2) * du                    # platform positions along track
scatterers = [(-40.0, 1.0), (0.0, 1.0), (35.0, 0.7)]     # azimuth offsets [m]

az_sig = np.zeros(n_pos, complex)
for x0, amp in scatterers:
    R_inst = np.sqrt(R0**2 + (u - x0)**2)
    az_sig += amp * np.exp(-1j*4*np.pi*R_inst/lam)
az_sig += 0.2*(rng.standard_normal(n_pos)+1j*rng.standard_normal(n_pos))

# azimuth matched filter: the reference chirp for a scatterer at x=0
R_ref = np.sqrt(R0**2 + u**2)
h_az = np.exp(-1j*4*np.pi*R_ref/lam)
image = np.abs(np.correlate(az_sig, h_az, "same"))
az_axis = u

plt.figure(figsize=(8.5, 2.6))
plt.plot(az_axis, image/image.max())
for x0, _ in scatterers: plt.axvline(x0, color="r", linestyle=":", linewidth=0.8)
plt.xlabel("azimuth [m]"); plt.title("azimuth compression: three scatterers resolved by a FLOWN aperture")
plt.tight_layout(); plt.show()
peaks = sig.find_peaks(image, height=image.max()*0.4)[0]
print("planted azimuths:", [s[0] for s in scatterers], " estimated:", np.round(az_axis[peaks], 1))

planted azimuths: [-40.0, 0.0, 35.0]  estimated: [-40.    0.   35.2]


/tmp/ipykernel_2990837/1125770396.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three scatterers recovered at **−40.0, 0.0, 35.2 m** in azimuth against planted values of −40, 0, 35 — resolved to a couple of metres at a range of 5 km, using an antenna the simulation never even specifies. The resolution came from the 205 m of *flight path*, not from any physical aperture.

**Why the along-track signal is a chirp.** As the platform passes a scatterer, the instantaneous range is $R(u) = \sqrt{R_0^2 + (u - x_0)^2}$, which near closest approach is approximately quadratic in $u$. Phase is $-4\pi R/\lambda$, so the recorded phase is quadratic in *position* — and quadratic phase is exactly what a chirp is. So azimuth compression is Session 1's matched filter applied along the track: `np.correlate(az_sig, h_az, "same")` is the same line of code, rotated ninety degrees. Range chirp compresses range; azimuth chirp compresses azimuth; multiply the two and you have an image.

Notice also that `h_az` is built for a scatterer at $x = 0$, yet it compresses all three. The chirp's shape depends on range, not on azimuth position, so one reference filter serves the whole line — which is precisely why the correlation works.

**The counterintuitive payoff.** Ask what happens to azimuth resolution as the platform flies *further from* the scene. The instinct is that it degrades. It does not: at greater range the target stays inside the real antenna's beam for a longer stretch of flight, so the synthetic aperture grows in proportion, and the two effects very nearly cancel. Strip-map SAR azimuth resolution is approximately $D/2$ — **half the physical antenna length**, independent of range.

Which yields a genuinely strange design rule: a *smaller* antenna gives *better* azimuth resolution, because a smaller antenna has a wider beam, which keeps each target illuminated over a longer synthetic aperture. That is the opposite of every optical intuition, and it is why radar satellites image the Earth at metre resolution from hundreds of kilometres up.

**What makes it fragile.** The whole method rests on phase coherence across positions. Platform position errors, oscillator drift, or atmospheric path variation destroy the phase relationship, and the synthetic aperture collapses back to the physical one. This is why SAR platforms carry precision navigation and why autofocus algorithms are standard — it is the same coherence requirement as Session 2's Doppler processing, transposed onto a spatial axis.

**And the honest scope of this demo.** It compresses in azimuth only, at a single range, with no range cell migration correction, no range–azimuth coupling, and no motion errors. Real SAR processors — range–Doppler, chirp scaling, back-projection — exist almost entirely to handle those. This cell shows the principle that makes SAR possible; the engineering that makes it work is a field in itself.

**The workshop in one line.** Bandwidth buys range resolution, pulse-to-pulse phase buys velocity, local noise estimation buys detection, and motion buys aperture — with the matched filter doing the work in three of the four.

## 6. Conclusion

Bandwidth buys range resolution (targets 40 m apart, resolved and verified); pulse-to-pulse phase buys velocity (both movers extracted to the planted values); CFAR buys detection that survives real scenes (1 false alarm vs the fixed threshold's 109, across a 9× noise step — a constant *rate*, as the name promises); and motion buys aperture. One curriculum's worth of tools, pointed at the sky.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — real apertures; [MIMO Communications](./MIMO_Communications.ipynb) — the comms twin.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — passive radar with a $30 dongle is a real (advanced) project.